# Load Dimension Tables From CSV

This notebook loads some dimension tables directly from csv files which are already curated and cleaned.  No need for a bronze load process for this data.

Starting the notebook with %run "/Workspace/Shared/notebook_init"  loads common variables and constants used across all notebooks for the Vinoworld project

CATALOG, BRONZE, SILVER, GOLD, AUDIT, RAW_FILES,
PIPELINE_RUN_ID, Utils, F, Row, datetime etc.

import uuid, time
from datetime import datetime, timezone
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, lit, input_file_name, col
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, IntegerType
sys.path.append("/Workspace/Shared")

import pipeline_utils as Utils
from pipeline_logging import pipeline_log_upsert, pipeline_step_log_upsert, ingestion_log_insert


In [0]:
%run "/Workspace/Shared/notebook_init"

In [0]:
# Imports and constants specific to the Arancione Bronze load.
# STORE_NAME identifies the source store; SOURCE_SUBPATH is the subfolder
# under RAW_FILES; TARGET_TABLE is the fully-qualified Bronze table name.

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from datetime import datetime, timezone
import traceback

%load_ext autoreload
%autoreload 2

SOURCE_SUBPATH = "masterdata"                       # case must match volume
SOURCE_PATH    = f"{RAW_FILES}{SOURCE_SUBPATH}"
TARGET_TABLE   = f"{SILVER}.dim_currency"

# print(f"Products Source Path:  {SOURCE_PATH}") 


In [0]:
# ----------------------------------------------------------------------
# Setup the variables need for initial call to pipeline_step_log_upsert
# ----------------------------------------------------------------------

nb = Utils.get_notebook_context(dbutils)
notebook_folder = nb['notebook_folder']
notebook_name = nb['notebook_name']

#logger.info(f"Inserting pipeline_step_log record for notebook {notebook_name}")

step_log_id       =  int(time.time() * 1_000_000)    
pipeline_run_id   = PIPELINE_RUN_ID
step_sequence     = 1
layer             = "silver"
target_table      = "vinoworld.silver dim_currency, dim_date, dim_region, dim_store, dim_territory"
status            = "running"
started_timestamp = datetime.now(timezone.utc)
rows_read         = 0
rows_written      = 0
error_message     = None


pipeline_step_log_upsert(spark, step_log_id, pipeline_run_id, step_sequence, notebook_folder, notebook_name, status, started_timestamp, layer, target_table)

# All Parameters
# pipeline_step_log_upsert(spark, step_log_id, pipeline_run_id, step_sequence, notebook_folder, notebook_name, status, started_timestamp, layer, target_table, rows_read, rows_written,  ended_timestamp,  error_message)

In [0]:
%skip
%sql

/* -----------------------------------------------------------------------
-- The raw datafiles for these are in pristine condition and can be reloaded without any processing
----------------------------------------------------------------------- */

TRUNCATE TABLE vinoworld.silver.dim_currency;
TRUNCATE TABLE vinoworld.silver.dim_date;
TRUNCATE TABLE vinoworld.silver.dim_region;
TRUNCATE TABLE vinoworld.silver.dim_store;
TRUNCATE TABLE vinoworld.silver.dim_territory;

In [0]:


TARGET_TABLE   = f"{SILVER}.dim_currency"

# Define the SQL logic for this specific dimension
currency_merge_sql = lambda t: f"""
    MERGE INTO {t} a
    USING temp_dim t
    ON a.CurrencyCode = t.CurrencyCode
    WHEN NOT MATCHED THEN 
    INSERT (CurrencyCode, CurrencyName, InsertedDate, UpdatedDate)
    VALUES (t.CurrencyCode, t.CurrencyName, t.InsertedDate, t.UpdatedDate)
"""

# Execute the helper
result = Utils.load_dim_from_csv(
    spark=spark,
    source_path=f"{SOURCE_PATH}/Currency.csv",
    target_table=TARGET_TABLE,
    merge_sql_fn=currency_merge_sql,
    add_timestamps=True  # This handles the current_timestamp() logic automatically
)

print(f"Results =  {result}")

In [0]:
TARGET_TABLE   = f"{SILVER}.dim_date"

# Define the SQL logic for this specific dimension
date_merge_sql = lambda t: f"""
  MERGE INTO {TARGET_TABLE} AS target
USING temp_dim AS source
ON target.YearMonth = source.YearMonth
WHEN MATCHED THEN
  UPDATE SET
    target.DateYear = source.DateYear,
    target.DateMonth = source.DateMonth,
    target.LastDayOfMonth = to_date(source.LastDayOfMonth, 'M/d/yyyy'),
    target.Quarter = source.Quarter,
    target.Season = source.Season
WHEN NOT MATCHED THEN
  INSERT (DateYear, DateMonth, YearMonth, LastDayOfMonth, Quarter, Season)
  VALUES (source.DateYear, source.DateMonth, source.YearMonth, to_date(source.LastDayOfMonth, 'M/d/yyyy'), source.Quarter, source.Season
  )
"""

# Execute the helper
result = Utils.load_dim_from_csv(
    spark=spark,
    source_path=f"{SOURCE_PATH}/Dates.csv",
    target_table=TARGET_TABLE,
    merge_sql_fn=date_merge_sql,
    add_timestamps=True  # This handles the current_timestamp() logic automatically
)

print(f"Results =  {result}")

In [0]:
TARGET_TABLE   = f"{SILVER}.dim_exchange_rate"

# Define the SQL logic for this specific dimension
er_merge_sql = lambda t: f"""
    MERGE INTO {TARGET_TABLE} a
            USING temp_dim t
            ON a.FromCurrency = t.FromCurrency
            AND a.ToCurrency = t.ToCurrency
            AND a.EffectiveDate = t.EffectiveDate
            WHEN MATCHED AND a.ExchangeRate <> t.AverageRate THEN
                UPDATE SET
                a.ExchangeRate = t.AverageRate,
                a.UpdatedDate = t.UpdatedDate
            WHEN NOT MATCHED THEN 
                INSERT (FromCurrency, ToCurrency, EffectiveDate, ExchangeRate, InsertedDate, UpdatedDate  )
                 VALUES ( t.FromCurrency, t.ToCurrency, t.EffectiveDate,  t.AverageRate,  t.InsertedDate,  t.UpdatedDate )  
            """
df_transform = lambda df: df.withColumn(
                            "EffectiveDate", F.to_timestamp("EffectiveDate", 'M/d/yyyy'))

# Execute the helper
result = Utils.load_dim_from_csv(
    spark=spark,
    source_path=f"{SOURCE_PATH}/ExchangeRates.csv",
    target_table=TARGET_TABLE,
    merge_sql_fn=er_merge_sql,
    add_timestamps=True,
    df_transform=df_transform
)

print(f"Results =  {result}")

In [0]:
TARGET_TABLE   = f"{SILVER}.dim_store"

# Define the SQL logic for this specific dimension
store_merge_sql = lambda t: f"""
     MERGE INTO {TARGET_TABLE} a
            USING temp_dim t
            ON a.StoreName = t.StoreName
            WHEN MATCHED AND (a.StoreType <> t.StoreType OR a.Description <> t.Description) THEN
                UPDATE SET
                a.StoreType = t.StoreType,
                a.Description = t.Description,
                a.UpdatedDate = t.UpdatedDate
            WHEN NOT MATCHED THEN 
                INSERT (StoreName, StoreType, Description, InsertedDate, UpdatedDate)
                VALUES (t.StoreName, t.StoreType, t.Description, t.InsertedDate, t.UpdatedDate)  
"""

# Execute the helper
result = Utils.load_dim_from_csv(
    spark=spark,
    source_path=f"{SOURCE_PATH}/Store.csv",
    target_table=TARGET_TABLE,
    merge_sql_fn=store_merge_sql,
    add_timestamps=True  # This handles the current_timestamp() logic automatically
)

print(f"Results =  {result}")

In [0]:
TARGET_TABLE   = f"{SILVER}.dim_territory"

# Define the SQL logic for this specific dimension
store_merge_sql = lambda t: f"""
   MERGE INTO {TARGET_TABLE} AS target
  USING temp_dim AS source
  ON target.TerritoryCode = source.TerritoryCode
  WHEN MATCHED THEN
    UPDATE SET
      target.TerritoryName = source.TerritoryName,
      target.TradeRegion   = source.TradeRegion,
      target.Continent     = source.Continent,
      target.UpdatedDate   = source.UpdatedDate
  WHEN NOT MATCHED THEN
      INSERT (TerritoryCode, TerritoryName, TradeRegion, Continent, InsertedDate, UpdatedDate)
      VALUES (source.TerritoryCode, source.TerritoryName, source.TradeRegion, source.Continent, source.InsertedDate, source.UpdatedDate)
"""
  
df_transform = lambda df: (
                            df.withColumn("InsertedDate", F.current_timestamp( ) )
                              .withColumn("UpdatedDate", F.current_timestamp())
                        )
  


# Execute the helper
result = Utils.load_dim_from_csv(
    spark=spark,
    source_path=f"{SOURCE_PATH}/Territory.csv",
    target_table=TARGET_TABLE,
    merge_sql_fn=store_merge_sql,
    add_timestamps=True,
     df_transform=df_transform
)

print(f"Results =  {result}")

In [0]:
%skip
%sql
SELECT 'dim_currency row count = '      || CAST(COUNT(*) AS STRING) FROM vinoworld.silver.dim_currency
UNION ALL
SELECT 'dim_date row count = '          || CAST(COUNT(*) AS STRING) FROM vinoworld.silver.dim_date
UNION ALL
SELECT 'dim_exchange_rate row count = ' || CAST(COUNT(*) AS STRING) FROM vinoworld.silver.dim_exchange_rate
UNION ALL
SELECT 'dim_store row count = '         || CAST(COUNT(*) AS STRING) FROM vinoworld.silver.dim_store
UNION ALL
SELECT 'dim_territory row count = '     || CAST(COUNT(*) AS STRING) FROM vinoworld.silver.dim_territory;

In [0]:
%skip


metrics = spark.sql(f"DESCRIBE HISTORY vinoworld.silver.dim_product LIMIT 1") \
               .select("operationMetrics") \
               .collect()[0][0]

rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
rows_updated  = int(metrics.get("numTargetRowsUpdated",  0))
rows_deleted  = int(metrics.get("numTargetRowsDeleted",  0))

print(f" Rows  Inserted: {rows_inserted:,}")
print(f" Rows     Updated: {rows_updated:,}"   )
